1/Thư viện

In [1]:
import numpy as np
from skimage import data

2/clip_histogram

In [2]:
def clip_histogram(hist, Beta):
    """Cắt và phân phối lại lược đồ
    hist: mảng (L,) số nguyên. Trả về mảng (L,) đã cắt, tổng không đổi."""
    hist = np.asarray(hist, dtype=np.int64)
    beta = max(1, int(round(Beta)))
    L = len(hist)

    clipped = np.clip(hist, 0, beta)
    excess = int(np.sum(hist - clipped))

    hist = clipped + excess // L
    remain = excess % L
    hist[:remain] += 1
    return hist

3/build_tile_luts

In [3]:
def build_tile_luts(img, grid=(8, 8), clip_limit=2.0):
    """Trả về mảng (Gy, Gx, L) uint8: bảng tra của từng ô """
    assert img.dtype == np.uint8 and img.ndim == 2
    H, W = img.shape

    Gy, Gx = grid
    if Gy > H or Gx > W or gy < 1 or Gx <1:
        raise ValueError("Lưới lớn hơn ảnh")
    
    L = 256
    
    luts = np.zeros((Gy, Gx, L), dtype=np.uint8)
    
    for i in range(Gy):
        for j in range(Gx):
            y_start, y_end = i * H // Gy, (i + 1) * H // Gy
            x_start, x_end = j * W // Gx, (j + 1) * W // Gx
            
            tile = img[y_start:y_end, x_start:x_end]
            N_tile = tile.size
            hist = np.bincount(tile.ravel(), minlength=L)

            # beta = N_tile/L * [1 + α/100 * (s_max - 1)] = N_tile/L * s_max)  (tương đương alpha=100, s_max=clip_limit trong công thức)
            beta = N_tile / L * clip_limit
            clipped_hist = clip_histogram(hist, beta)
            
            cdf = np.cumsum(clipped_hist)
            luts[i, j] = np.round((L - 1) * cdf / cdf[-1]).astype(np.uint8)
            
    return luts

4/Clahe

In [4]:
def clahe(img, grid=(8, 8), clip_limit=2.0):
    """CLAHE đầy đủ. img: (H, W) uint8. Trả về (H, W) uint8 """
    assert img.dtype == np.uint8 and img.ndim == 2
    H, W = img.shape
    Gy, Gx = grid
    luts = build_tile_luts(img, grid, clip_limit)
    
    out = np.zeros_like(img, dtype=np.float64)
    tile_h = H / Gy
    tile_w = W / Gx
    
    for y in range(H):
        for x in range(W):
            fy = (y + 0.5) / tile_h - 0.5
            fx = (x + 0.5) / tile_w - 0.5
            
            i = int(fy//1)
            j = int(fx//1)
            
            a = fx - j
            b = fy - i
            
            i0 = max(i, 0);  i1 = min(i + 1, Gy - 1)
            j0 = max(j, 0);  j1 = min(j + 1, Gx - 1)
            
            v = img[y, x]
            
            T00 = luts[i0, j0, v]
            T01 = luts[i0, j1, v]
            T10 = luts[i1, j0, v]
            T11 = luts[i1, j1, v]

            T_top = (1 - a) * T00 + a * T01
            T_bot = (1 - a) * T10 + a * T11

            val = (1 - b) * T_top + b * T_bot
            out[y, x] = val
            
    return np.rint(out).astype(np.uint8)